Load and Inspect Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset (adjust path if running from notebooks/ directory)
df = pd.read_csv('../creditcard.csv')

print("Dataset Shape:", df.shape)
print("\nClass Distribution:")
print(df['Class'].value_counts(normalize=True))

# Quick check for missing values
print("\nMissing Values:", df.isnull().sum().max())

Leakage-Safe Train-Test Split & Scaling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separate features and target
X = df.drop(columns=['Class'])
y = df['Class']

# Train-test split (Stratified to maintain the rare fraud ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale 'Time' and 'Amount' features (Fit on train only to prevent data leakage)
scaler = StandardScaler()
X_train[['Time', 'Amount']] = scaler.fit_transform(X_train[['Time', 'Amount']])
X_test[['Time', 'Amount']] = scaler.transform(X_test[['Time', 'Amount']])

Handle Class Imbalance with SMOTE (Training Data Only)

In [ ]:
from imblearn.over_sampling import SMOTE

# Apply SMOTE strictly on training data
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Before SMOTE shape:", X_train.shape)
print("After SMOTE shape:", X_train_resampled.shape)

Train and Evaluate Baseline Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

for name, model in models.items():
    print(f"--- Training {name} ---")
    model.fit(X_train_resampled, y_train_resampled)
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
    print(f"PR-AUC: {average_precision_score(y_test, y_prob):.4f}\n")

Save Best Model & Scaler Artifacts

In [ ]:
import joblib
import os

# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Assuming Random Forest or your best-performing model is selected
best_model = models["Random Forest"]

# Save artifacts for the Streamlit app
joblib.dump(best_model, '../models/fraud_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

print("Model and scaler saved successfully in /models directory!")